# Introduction to the ODL tomography interface
In this lab we will perform our first tomographic reconstruction. The problem involves reconstructing a given 2D image from its sinogram, obtained with the forward operator given by parallel ray transforms across all angles. 

1. First, define the function space. Our function will be defined in the space $L_2([-2, 2]^2)$ over the real numbers. Use the function ``odl.uniform_discr`` to create a discretized space of real functions in two variables, i.e., images, and pass ``dtype='float32'``, which the ASTRA backend requires in ODL 1.0.

2. Define the unit disk indicator function (i.e., $x^2 + y^2 \leq 1$) with an appropriate Python function and turn it into an element of the space. This can be a bit tricky since you cannot use ``if`` in your function, but it may help to know that ``True`` equals 1 in Python.

3. Use the method ``show`` to visualize the function. Since our space is two-dimensional, ``show`` will interpret a space element as grey-scale image. 

4. Next we set up the tomography geometry. For a parallel-beam geometry, use the function [odl.applications.tomo.parallel_beam_geometry](https://odl.readthedocs.io/generated/odl.applications.tomo.geometry.parallel.parallel_beam_geometry.html), which automatically sets the angles and parallel beam distances in order to fully sample the discretized function.

5. Now we can form the (discretized) forward operator (the measurement matrix) by calling [odl.applications.tomo.RayTransform](https://odl.readthedocs.io/generated/odl.applications.tomo.operators.ray_trafo.RayTransform.html) with the space and the geometry.

6. Obtain a sinogram by applying the forward operator to the space element and display it with the ``show`` method. Note that this sinogram is analytically tractable in the corresponding undiscretized/infinite-dimensional problem, so you should verify that the numerical sinogram has the correct shape.

7. Add some [Gaussian noise](https://odl.readthedocs.io/generated/odl.core.phantom.noise.white_noise.html) to your sinogram (which lives in the space ``Operator.range``) and compare with the original sinogram.

8. Next, use the adjoint of the operator to backproject the noisy sinogram, which should be accessible with ``Operator.adjoint``. Compare with the original image and try to figure out why the backprojection looks the way it does.

9. A filtered backprojection can be obtained using [odl.applications.tomo.fbp_op](https://odl.readthedocs.io/generated/odl.applications.tomo.analytic.filtered_back_projection.fbp_op.html). You should test different filter types by passing ``filter_type``. Compare with the original image.

10. Play around with this example by changing the number of angles, that is, by setting the ``num_angles``-parameter when defining the geometry, or by using a different geometry, or by using a different image like the [Shepp–Logan phantom](https://odl.readthedocs.io/generated/odl.core.phantom.transmission.shepp_logan.html), etc. Observe the characteristic shapes of the artifacts.

In [ ]:
import odl
import numpy as np

# 1. Define the 2D function space L_2([-2, 2]^2)
# We use shape=(300, 300) for a standard resolution, passing float32 for ASTRA backend compatibility.
space = odl.uniform_discr(min_pt=[-2, -2], max_pt=[2, 2], shape=(300, 300), dtype='float32')

# 2. Define the unit disk indicator function
# The lambda function receives the spatial coordinates as an array x. 
# x[0] is the x-axis, x[1] is the y-axis. The boolean output automatically casts True to 1 and False to 0.
disk = space.element(lambda x: x[0]**2 + x[1]**2 <= 1.0)

# 3. Visualize the function
disk.show(title="1. Original Unit Disk", force_show=True)

# 4. Set up the parallel-beam geometry
# By just passing the space, ODL automatically sets angles and detector spacing for full sampling.
geometry = odl.tomo.parallel_beam_geometry(space)

# 5. Form the forward operator (Ray Transform / Measurement Matrix)
ray_trafo = odl.tomo.RayTransform(space, geometry)

# 6. Apply forward operator to get the sinogram
# The analytical sinogram of a uniform disk should look like a solid semi-ellipse.
sinogram = ray_trafo(disk)
sinogram.show(title="2. Sinogram (Forward Transform)", force_show=True)

# 7. Add Gaussian noise to the sinogram
# We scale the noise relative to the maximum value of the sinogram so it doesn't wash out the signal entirely.
noise = odl.phantom.noise.white_noise(ray_trafo.range)
noise_level = 0.05 * odl.max(odl.abs(sinogram)) 
noisy_sinogram = sinogram + noise_level * noise
noisy_sinogram.show(title="3. Noisy Sinogram", force_show=True)

# 8. Adjoint (Unfiltered Backprojection)
backprojected = ray_trafo.adjoint(noisy_sinogram)
backprojected.show(title="4. Unfiltered Backprojection (Adjoint)", force_show=True)

# 9. Filtered Backprojection (FBP)
# We instantiate two different FBP operators to test filter types. 
# 'Ram-Lak' preserves edges but amplifies noise. 'Hann' smooths out noise at the cost of slight blurring.
fbp_ramlak = odl.tomo.fbp_op(ray_trafo, filter_type='Ram-Lak')
fbp_hann = odl.tomo.fbp_op(ray_trafo, filter_type='Hann')

recon_ramlak = fbp_ramlak(noisy_sinogram)
recon_hann = fbp_hann(noisy_sinogram)

recon_ramlak.show(title="5a. FBP Reconstruction (Ram-Lak)", force_show=True)
recon_hann.show(title="5b. FBP Reconstruction (Hann)", force_show=True)

# 10. Experimenting: Shepp-Logan Phantom and Sparse Angles
# Redefining the geometry with far fewer angles (e.g., 30 instead of the automatic ~300)
sparse_geometry = odl.tomo.parallel_beam_geometry(space, num_angles=30)
sparse_ray_trafo = odl.tomo.RayTransform(space, sparse_geometry)

# Generating the Shepp-Logan phantom
shepp_logan = odl.phantom.shepp_logan(space, modified=True)
sparse_sinogram = sparse_ray_trafo(shepp_logan)

# Sparse FBP Reconstruction
sparse_fbp = odl.tomo.fbp_op(sparse_ray_trafo, filter_type='Hann')
sparse_recon = sparse_fbp(sparse_sinogram)

shepp_logan.show(title="6. Shepp-Logan Phantom", force_show=True)
sparse_recon.show(title="7. Sparse Angle FBP Reconstruction (30 angles)", force_show=True)